In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("NYC Mobility - Data Exploration")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/16 18:34:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.4


In [2]:
file_path = "../data/raw/yellow_tripdata_2025-01.parquet"

trips = spark.read.parquet(file_path)

In [3]:
trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [4]:
trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
row_count = trips.count()
print(f"Anzahl Fahrten: {row_count:,}")

Anzahl Fahrten: 3,475,226


In [6]:
null_counts = trips.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in trips.columns
])

null_counts.show(vertical=True)

[Stage 6:====================================>                      (5 + 3) / 8]

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 540149 
 trip_distance         | 0      
 RatecodeID            | 540149 
 store_and_fwd_flag    | 540149 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 540149 
 Airport_fee           | 540149 
 cbd_congestion_fee    | 0      



In [7]:
null_percentages = trips.select([
    (
        F.sum(F.col(c).isNull().cast("int")) / row_count * 100
    ).alias(c)
    for c in trips.columns
])

null_percentages.show(vertical=True)

[Stage 9:====================================>                      (5 + 3) / 8]

-RECORD 0----------------------------------
 VendorID              | 0.0               
 tpep_pickup_datetime  | 0.0               
 tpep_dropoff_datetime | 0.0               
 passenger_count       | 15.54284527107014 
 trip_distance         | 0.0               
 RatecodeID            | 15.54284527107014 
 store_and_fwd_flag    | 15.54284527107014 
 PULocationID          | 0.0               
 DOLocationID          | 0.0               
 payment_type          | 0.0               
 fare_amount           | 0.0               
 extra                 | 0.0               
 mta_tax               | 0.0               
 tip_amount            | 0.0               
 tolls_amount          | 0.0               
 improvement_surcharge | 0.0               
 total_amount          | 0.0               
 congestion_surcharge  | 15.54284527107014 
 Airport_fee           | 15.54284527107014 
 cbd_congestion_fee    | 0.0               



In [8]:
trips.select(
    F.min("trip_distance").alias("min_trip_distance"),
    F.max("trip_distance").alias("max_trip_distance"),
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.min("passenger_count").alias("min_passengers"),
    F.max("passenger_count").alias("max_passengers"),
).show()

+-----------------+-----------------+--------+---------+--------------+--------------+
|min_trip_distance|max_trip_distance|min_fare| max_fare|min_passengers|max_passengers|
+-----------------+-----------------+--------+---------+--------------+--------------+
|              0.0|        276423.57|  -900.0|863372.12|             0|             9|
+-----------------+-----------------+--------+---------+--------------+--------------+



In [9]:
trips.select(
    F.min("tpep_pickup_datetime").alias("first_pickup"),
    F.max("tpep_pickup_datetime").alias("last_pickup"),
    F.min("tpep_dropoff_datetime").alias("first_dropoff"),
    F.max("tpep_dropoff_datetime").alias("last_dropoff"),
).show()

+-------------------+-------------------+-------------------+-------------------+
|       first_pickup|        last_pickup|      first_dropoff|       last_dropoff|
+-------------------+-------------------+-------------------+-------------------+
|2024-12-31 20:47:55|2025-02-01 00:00:44|2024-12-18 07:52:40|2025-02-01 23:44:11|
+-------------------+-------------------+-------------------+-------------------+



In [10]:
from datetime import datetime

january_start = datetime(2025, 1, 1)
february_start = datetime(2025, 2, 1)

trips.select(
    F.sum(
        (F.col("tpep_pickup_datetime") < january_start).cast("int")
    ).alias("pickup_before_january"),

    F.sum(
        (F.col("tpep_pickup_datetime") >= february_start).cast("int")
    ).alias("pickup_after_january"),

    F.sum(
        (F.col("tpep_dropoff_datetime") < january_start).cast("int")
    ).alias("dropoff_before_january"),

    F.sum(
        (F.col("tpep_dropoff_datetime") >= february_start).cast("int")
    ).alias("dropoff_after_january")
).show()

+---------------------+--------------------+----------------------+---------------------+
|pickup_before_january|pickup_after_january|dropoff_before_january|dropoff_after_january|
+---------------------+--------------------+----------------------+---------------------+
|                   21|                   1|                    14|                 1797|
+---------------------+--------------------+----------------------+---------------------+



In [11]:
invalid_duration_count = trips.filter(
    F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")
).count()

print(f"Dropoff vor Pickup: {invalid_duration_count:,}")

Dropoff vor Pickup: 124


In [12]:
trips_profiled = trips.withColumn(
    "trip_duration_minutes",
    F.timestamp_diff(
        "MINUTE",
        F.col("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime")
    )
)

In [13]:
trips_profiled.select(
    F.min("trip_duration_minutes").alias("min_duration"),
    F.max("trip_duration_minutes").alias("max_duration"),
    F.avg("trip_duration_minutes").alias("avg_duration")
).show()

[Stage 24:====================================>                     (5 + 3) / 8]

+------------+------------+------------------+
|min_duration|max_duration|      avg_duration|
+------------+------------+------------------+
|      -51472|        5626|14.534479484211962|
+------------+------------+------------------+



In [14]:
quantiles = trips_profiled.approxQuantile(
    "trip_duration_minutes",
    [0.01, 0.25, 0.50, 0.75, 0.95, 0.99],
    0.001
)

print("1%: ", quantiles[0])
print("25%:", quantiles[1])
print("50%:", quantiles[2])
print("75%:", quantiles[3])
print("95%:", quantiles[4])
print("99%:", quantiles[5])

1%:  0.0
25%: 7.0
50%: 11.0
75%: 18.0
95%: 36.0
99%: 58.0


In [15]:
trips_profiled.select(
    F.sum((F.col("trip_duration_minutes") < 0).cast("int")).alias("negative"),
    F.sum((F.col("trip_duration_minutes") == 0).cast("int")).alias("zero"),
    F.sum((F.col("trip_duration_minutes") > 60).cast("int")).alias("over_60"),
    F.sum((F.col("trip_duration_minutes") > 120).cast("int")).alias("over_120"),
    F.sum((F.col("trip_duration_minutes") > 180).cast("int")).alias("over_180")
).show()

[Stage 29:====================================>                     (5 + 3) / 8]

+--------+-----+-------+--------+--------+
|negative| zero|over_60|over_120|over_180|
+--------+-----+-------+--------+--------+
|       6|40105|  29489|    1981|    1374|
+--------+-----+-------+--------+--------+



### Findings – Trip Duration

- 124 records have a dropoff timestamp earlier than the pickup timestamp and are considered invalid.
- 99% of trips have a duration of 58 minutes or less.
- 29,489 trips exceed 60 minutes, but long trips are not automatically considered invalid.
- Extreme durations of several hours exist and require further investigation.
- Zero-minute durations are not automatically removed because `timestamp_diff("MINUTE")` truncates sub-minute durations.

In [16]:
distance_quantiles = trips.approxQuantile(
    "trip_distance",
    [0.01, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999],
    0.001
)

for q, value in zip(
    ["1%", "25%", "50%", "75%", "95%", "99%", "99.9%"],
    distance_quantiles
):
    print(f"{q}: {value}")

[Stage 32:====================================>                     (5 + 3) / 8]

1%: 0.0
25%: 0.98
50%: 1.67
75%: 3.1
95%: 11.82
99%: 19.51
99.9%: 276423.57


In [17]:
trips.select(
    F.sum((F.col("trip_distance") == 0).cast("int")).alias("zero_distance"),
    F.sum((F.col("trip_distance") < 0).cast("int")).alias("negative_distance"),
    F.sum((F.col("trip_distance") > 50).cast("int")).alias("over_50"),
    F.sum((F.col("trip_distance") > 100).cast("int")).alias("over_100"),
    F.sum((F.col("trip_distance") > 500).cast("int")).alias("over_500")
).show()

[Stage 34:====================================>                     (5 + 3) / 8]

+-------------+-----------------+-------+--------+--------+
|zero_distance|negative_distance|over_50|over_100|over_500|
+-------------+-----------------+-------+--------+--------+
|        90893|                0|    533|     162|     118|
+-------------+-----------------+-------+--------+--------+



In [18]:
trips.groupBy("trip_distance") \
    .count() \
    .orderBy(F.desc("trip_distance")) \
    .show(20, truncate=False)

[Stage 37:====================================>                     (5 + 3) / 8]

+-------------+-----+
|trip_distance|count|
+-------------+-----+
|276423.57    |1    |
|276099.95    |1    |
|222167.49    |1    |
|206137.99    |1    |
|202771.63    |1    |
|189687.43    |1    |
|181139.99    |1    |
|168079.57    |1    |
|167452.94    |1    |
|164959.95    |1    |
|158925.09    |1    |
|156037.94    |1    |
|143712.27    |1    |
|135116.83    |1    |
|134033.15    |1    |
|124083.23    |1    |
|121799.97    |1    |
|121555.16    |1    |
|118927.12    |1    |
|118435.89    |1    |
+-------------+-----+
only showing top 20 rows


In [19]:
trips.filter(

    F.col("trip_distance") == 276423.57

).select(

    "tpep_pickup_datetime",

    "tpep_dropoff_datetime",

    "PULocationID",

    "DOLocationID",

    "trip_distance",

    "fare_amount",

    "total_amount"

).show(20, truncate=False)

+--------------------+---------------------+------------+------------+-------------+-----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|fare_amount|total_amount|
+--------------------+---------------------+------------+------------+-------------+-----------+------------+
|2025-01-22 21:04:00 |2025-01-22 21:13:00  |161         |170         |276423.57    |-4.75      |5.0         |
+--------------------+---------------------+------------+------------+-------------+-----------+------------+



In [20]:
trips.filter(
    F.col("trip_distance") > 100
).select(
    "trip_distance",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount"
).orderBy("trip_distance").show(30, truncate=False)

+-------------+--------------------+---------------------+------------+------------+-----------+
|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|fare_amount|
+-------------+--------------------+---------------------+------------+------------+-----------+
|100.17       |2025-01-22 16:19:30 |2025-01-22 18:22:49  |132         |265         |400.0      |
|100.69       |2025-01-31 00:48:25 |2025-01-31 02:52:26  |161         |265         |300.63     |
|101.56       |2025-01-07 10:26:50 |2025-01-07 12:10:35  |132         |265         |410.0      |
|102.2        |2025-01-31 12:32:28 |2025-01-31 12:53:16  |141         |88          |31.7       |
|102.44       |2025-01-20 07:30:05 |2025-01-20 10:16:55  |216         |230         |250.0      |
|104.21       |2025-01-21 10:35:49 |2025-01-21 12:31:34  |219         |265         |200.0      |
|105.24       |2025-01-01 08:44:11 |2025-01-01 10:12:44  |246         |265         |300.0      |
|108.9        |2025-01-05 18:2

In [21]:
total_rows = trips.count()
distinct_rows = trips.distinct().count()

print(f"Gesamt:            {total_rows:,}")
print(f"Eindeutige Zeilen: {distinct_rows:,}")
print(f"Exakte Duplikate:  {total_rows - distinct_rows:,}")

[Stage 49:================================>                         (5 + 4) / 9]

Gesamt:            3,475,226
Eindeutige Zeilen: 3,475,226
Exakte Duplikate:  0


In [22]:
trip_keys = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance"
]

possible_duplicates = (
    trips
    .groupBy(*trip_keys)
    .agg(
        F.count("*").alias("records"),
        F.collect_set("fare_amount").alias("fare_amounts"),
        F.collect_set("total_amount").alias("total_amounts")
    )
    .filter(F.col("records") > 1)
    .orderBy(F.desc("records"))
)

possible_duplicates.show(30, truncate=False)

[Stage 55:>                                                         (0 + 8) / 9]

+--------+--------------------+---------------------+------------+------------+-------------+-------+-------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|records|fare_amounts |total_amounts    |
+--------+--------------------+---------------------+------------+------------+-------------+-------+-------------+-----------------+
|2       |2025-01-15 10:08:42 |2025-01-15 10:13:06  |48          |50          |1.06         |3      |[0.0]        |[-4.0, 4.0, 4.75]|
|2       |2025-01-22 07:35:30 |2025-01-22 07:46:44  |100         |158         |1.57         |3      |[0.0]        |[-4.0, 4.0, 4.75]|
|2       |2025-01-01 00:02:49 |2025-01-01 00:06:00  |75          |75          |0.48         |2      |[-5.1, 5.1]  |[-7.6, 7.6]      |
|2       |2025-01-01 00:08:40 |2025-01-01 00:16:01  |132         |203         |4.58         |2      |[19.1, -19.1]|[21.6, -21.6]    |
|2       |2025-01-01 00:08:22 |2025-01-01 00:10:16  |138      

In [23]:
duplicate_summary = (
    trips
    .groupBy(*trip_keys)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_groups = duplicate_summary.count()

duplicate_records = (
    duplicate_summary
    .select(
        F.sum("count").alias("records")
    )
    .first()["records"]
)

print(f"Gruppen mit mehreren Records: {duplicate_groups:,}")
print(f"Records in diesen Gruppen:    {duplicate_records:,}")

[Stage 64:>                                                         (0 + 8) / 9]

Gruppen mit mehreren Records: 52,531
Records in diesen Gruppen:    105,064


In [24]:
extra_records = (
    duplicate_summary
    .select(
        F.sum(F.col("count") - 1).alias("extra_records")
    )
    .first()["extra_records"]
)

print(f"Potenzielle zusätzliche Records: {extra_records:,}")

[Stage 68:====================================>                     (5 + 3) / 8]

Potenzielle zusätzliche Records: 52,533


In [25]:
(
    duplicate_summary
    .groupBy("VendorID")
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.sum("count").alias("records")
    )
    .orderBy("VendorID")
    .show()
)

[Stage 76:>                                                         (0 + 8) / 9]

+--------+----------------+-------+
|VendorID|duplicate_groups|records|
+--------+----------------+-------+
|       2|           52531| 105064|
+--------+----------------+-------+



### Finding – Potential Correction / Reversal Records

- 52,531 groups contain multiple records with the same constructed trip key.
- These groups contain 105,064 records in total, corresponding to 52,533 potential additional records.
- All affected groups belong to `VendorID = 2`.
- Many groups contain matching positive and negative fare amounts, indicating possible correction or reversal transactions.
- Since the dataset does not provide a unique trip identifier, these records are not automatically removed at this stage.
- Their treatment must be considered specifically when constructing the demand target to avoid potentially counting accounting corrections as additional taxi demand.

In [26]:
null_columns = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

all_null = trips.filter(
    F.col("passenger_count").isNull()
    & F.col("RatecodeID").isNull()
    & F.col("store_and_fwd_flag").isNull()
    & F.col("congestion_surcharge").isNull()
    & F.col("Airport_fee").isNull()
)

print(f"Zeilen mit allen fünf NULL: {all_null.count():,}")

Zeilen mit allen fünf NULL: 540,149


In [27]:
all_null.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "payment_type",
    "fare_amount",
    "total_amount"
).show(20, truncate=False)

+--------+--------------------+---------------------+------------+------------+-------------+------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|payment_type|fare_amount|total_amount|
+--------+--------------------+---------------------+------------+------------+-------------+------------+-----------+------------+
|2       |2025-01-01 00:39:17 |2025-01-01 01:10:42  |48          |42          |6.75         |0           |2.32       |14.9        |
|2       |2025-01-01 00:33:40 |2025-01-01 01:10:34  |4           |48          |3.88         |0           |6.02       |10.02       |
|1       |2025-01-01 00:09:32 |2025-01-01 00:12:38  |158         |158         |0.0          |0           |22.72      |26.72       |
|2       |2025-01-01 00:21:42 |2025-01-01 00:36:29  |137         |211         |3.55         |0           |-2.35      |5.8         |
|2       |2025-01-01 00:56:04 |2025-01-01 01:06:11  |140         |233       

In [28]:
all_null.groupBy("VendorID") \
    .count() \
    .orderBy("VendorID") \
    .show()

+--------+------+
|VendorID| count|
+--------+------+
|       1| 88204|
|       2|451456|
|       6|   489|
+--------+------+



In [29]:
all_null.groupBy("payment_type") \
    .count() \
    .orderBy("payment_type") \
    .show()

+------------+------+
|payment_type| count|
+------------+------+
|           0|540149|
+------------+------+



### Finding – Structured Missing Values

- 540,149 records have simultaneous missing values in `passenger_count`,
  `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee`.
- All 540,149 records have `payment_type = 0`.
- The missing values therefore follow a structured pattern rather than occurring
  independently across the dataset.
- These records retain the pickup timestamp and pickup location required for the
  demand-prediction use case.
- They will therefore not be removed solely because of these missing values.

## 3. Demand Dataset Preparation

Based on the findings from the exploratory data analysis, the raw trip data
is cleaned specifically for the demand-prediction use case.

The target variable represents the number of valid taxi pickups per
pickup zone and hour.

In [31]:
demand_base = trips.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "total_amount"
)

In [36]:
demand_clean = demand_base.filter(
    (F.col("tpep_pickup_datetime") >= F.lit("2025-01-01 00:00:00")) &
    (F.col("tpep_pickup_datetime") < F.lit("2025-02-01 00:00:00"))
)

In [37]:
print(f"Vorher: {demand_base.count():,}")
print(f"Nach Datumsfilter: {demand_clean.count():,}")

Vorher: 3,475,226
Nach Datumsfilter: 3,475,204


In [38]:
demand_clean = demand_clean.filter(
    F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime")
)

In [39]:
demand_clean = demand_clean.filter(
    F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime")
)

In [40]:
print(f"Nach Zeitvalidierung: {demand_clean.count():,}")

Nach Zeitvalidierung: 3,475,080


In [41]:
demand_clean.select(
    F.min("PULocationID").alias("min_zone"),
    F.max("PULocationID").alias("max_zone"),
    F.countDistinct("PULocationID").alias("distinct_zones")
).show()

[Stage 113:===================================>                     (5 + 3) / 8]

+--------+--------+--------------+
|min_zone|max_zone|distinct_zones|
+--------+--------+--------------+
|       1|     265|           261|
+--------+--------+--------------+



In [42]:
demand_clean.groupBy("PULocationID") \
    .count() \
    .orderBy("PULocationID") \
    .show(300)

+------------+------+
|PULocationID| count|
+------------+------+
|           1|   377|
|           2|     6|
|           3|   175|
|           4|  7479|
|           5|     3|
|           6|    87|
|           7|  3190|
|           8|    22|
|           9|   117|
|          10|  1329|
|          11|   159|
|          12|   946|
|          13| 21346|
|          14|   940|
|          15|    97|
|          16|   120|
|          17|  1551|
|          18|   344|
|          19|   234|
|          20|   199|
|          21|   310|
|          22|   343|
|          23|    15|
|          24|  8576|
|          25|  2353|
|          26|   684|
|          27|     4|
|          28|   696|
|          29|   270|
|          30|    12|
|          31|    37|
|          32|   220|
|          33|  2669|
|          34|   257|
|          35|  1445|
|          36|  1218|
|          37|  1820|
|          38|   210|
|          39|  1826|
|          40|   813|
|          41| 11037|
|          42|  5981|
|         

In [44]:
correction_groups = (
    demand_clean
    .groupBy(*trip_keys)
    .agg(
        F.count("*").alias("records"),
        F.min("fare_amount").alias("min_fare"),
        F.max("fare_amount").alias("max_fare")
    )
    .filter(
        (F.col("records") > 1) &
        (F.col("min_fare") < 0) &
        (F.col("max_fare") >= 0)
    )
)

print(
    f"Potenzielle Correction/Reversal-Gruppen: "
    f"{correction_groups.count():,}"
)

[Stage 127:>                                                        (0 + 8) / 9]

Potenzielle Correction/Reversal-Gruppen: 52,235


In [45]:
correction_groups.select(
    F.sum(F.col("records") - 1).alias("potential_extra_records")
).show()

[Stage 133:>                                                        (0 + 8) / 9]

+-----------------------+
|potential_extra_records|
+-----------------------+
|                  52235|
+-----------------------+



In [46]:
print(
    f"Correction/Reversal-Gruppen: "
    f"{correction_groups.count():,}"
)

[Stage 139:>                                                        (0 + 8) / 9]

Correction/Reversal-Gruppen: 52,235


In [47]:
correction_keys = correction_groups.select(*trip_keys)

demand_valid = demand_clean.join(
    correction_keys,
    on=trip_keys,
    how="left_anti"
)

In [48]:
before_corrections = demand_clean.count()
after_corrections = demand_valid.count()

print(f"Vor Correction-Filter:  {before_corrections:,}")
print(f"Nach Correction-Filter: {after_corrections:,}")
print(f"Entfernt:                {before_corrections - after_corrections:,}")

[Stage 154:===================================>                     (5 + 3) / 8]

Vor Correction-Filter:  3,475,080
Nach Correction-Filter: 3,370,610
Entfernt:                104,470


In [49]:
demand_valid = demand_valid.select(
    "tpep_pickup_datetime",
    "PULocationID"
)

In [50]:
demand_valid = demand_valid.withColumn(
    "pickup_hour",
    F.date_trunc(
        "hour",
        F.col("tpep_pickup_datetime")
    )
)

In [51]:
demand_valid.select(
    "tpep_pickup_datetime",
    "pickup_hour",
    "PULocationID"
).show(10, truncate=False)

[Stage 161:>                                                        (0 + 8) / 9]

+--------------------+-------------------+------------+
|tpep_pickup_datetime|pickup_hour        |PULocationID|
+--------------------+-------------------+------------+
|2025-01-01 00:44:59 |2025-01-01 00:00:00|261         |
|2025-01-01 00:07:44 |2025-01-01 00:00:00|249         |
|2025-01-01 00:30:46 |2025-01-01 00:00:00|234         |
|2025-01-01 00:45:36 |2025-01-01 00:00:00|143         |
|2025-01-01 00:49:18 |2025-01-01 00:00:00|140         |
|2025-01-01 00:29:12 |2025-01-01 00:00:00|262         |
|2025-01-01 00:48:37 |2025-01-01 00:00:00|79          |
|2025-01-01 00:36:51 |2025-01-01 00:00:00|141         |
|2025-01-01 00:00:17 |2025-01-01 00:00:00|234         |
|2025-01-01 00:44:37 |2025-01-01 00:00:00|209         |
+--------------------+-------------------+------------+
only showing top 10 rows


In [52]:
hourly_demand = (
    demand_valid
    .groupBy(
        "PULocationID",
        "pickup_hour"
    )
    .agg(
        F.count("*").alias("demand")
    )
    .orderBy(
        "pickup_hour",
        "PULocationID"
    )
)

In [53]:
hourly_demand.show(30, truncate=False)

[Stage 177:=================================================>       (7 + 1) / 8]

+------------+-------------------+------+
|PULocationID|pickup_hour        |demand|
+------------+-------------------+------+
|4           |2025-01-01 00:00:00|28    |
|7           |2025-01-01 00:00:00|12    |
|9           |2025-01-01 00:00:00|1     |
|10          |2025-01-01 00:00:00|1     |
|12          |2025-01-01 00:00:00|1     |
|13          |2025-01-01 00:00:00|7     |
|14          |2025-01-01 00:00:00|4     |
|15          |2025-01-01 00:00:00|1     |
|16          |2025-01-01 00:00:00|1     |
|17          |2025-01-01 00:00:00|5     |
|18          |2025-01-01 00:00:00|2     |
|22          |2025-01-01 00:00:00|3     |
|24          |2025-01-01 00:00:00|22    |
|25          |2025-01-01 00:00:00|12    |
|26          |2025-01-01 00:00:00|1     |
|28          |2025-01-01 00:00:00|2     |
|33          |2025-01-01 00:00:00|18    |
|36          |2025-01-01 00:00:00|6     |
|37          |2025-01-01 00:00:00|10    |
|39          |2025-01-01 00:00:00|1     |
|40          |2025-01-01 00:00:00|

In [55]:
demand_valid = demand_valid.select(
    "tpep_pickup_datetime",
    "PULocationID"
)

In [56]:
demand_valid = demand_valid.withColumn(
    "pickup_hour",
    F.date_trunc(
        "hour",
        F.col("tpep_pickup_datetime")
    )
)

In [57]:
demand_valid.show(10, truncate=False)

[Stage 184:>                                                        (0 + 8) / 9]

+--------------------+------------+-------------------+
|tpep_pickup_datetime|PULocationID|pickup_hour        |
+--------------------+------------+-------------------+
|2025-01-01 00:44:59 |261         |2025-01-01 00:00:00|
|2025-01-01 00:07:44 |249         |2025-01-01 00:00:00|
|2025-01-01 00:30:46 |234         |2025-01-01 00:00:00|
|2025-01-01 00:45:36 |143         |2025-01-01 00:00:00|
|2025-01-01 00:49:18 |140         |2025-01-01 00:00:00|
|2025-01-01 00:29:12 |262         |2025-01-01 00:00:00|
|2025-01-01 00:48:37 |79          |2025-01-01 00:00:00|
|2025-01-01 00:36:51 |141         |2025-01-01 00:00:00|
|2025-01-01 00:00:17 |234         |2025-01-01 00:00:00|
|2025-01-01 00:44:37 |209         |2025-01-01 00:00:00|
+--------------------+------------+-------------------+
only showing top 10 rows


In [58]:
hourly_demand = (
    demand_valid
    .groupBy(
        "PULocationID",
        "pickup_hour"
    )
    .agg(
        F.count("*").alias("demand")
    )
)

In [59]:
hourly_demand.orderBy(
    "pickup_hour",
    "PULocationID"
).show(30, truncate=False)

[Stage 200:===================================>                     (5 + 3) / 8]

+------------+-------------------+------+
|PULocationID|pickup_hour        |demand|
+------------+-------------------+------+
|4           |2025-01-01 00:00:00|28    |
|7           |2025-01-01 00:00:00|12    |
|9           |2025-01-01 00:00:00|1     |
|10          |2025-01-01 00:00:00|1     |
|12          |2025-01-01 00:00:00|1     |
|13          |2025-01-01 00:00:00|7     |
|14          |2025-01-01 00:00:00|4     |
|15          |2025-01-01 00:00:00|1     |
|16          |2025-01-01 00:00:00|1     |
|17          |2025-01-01 00:00:00|5     |
|18          |2025-01-01 00:00:00|2     |
|22          |2025-01-01 00:00:00|3     |
|24          |2025-01-01 00:00:00|22    |
|25          |2025-01-01 00:00:00|12    |
|26          |2025-01-01 00:00:00|1     |
|28          |2025-01-01 00:00:00|2     |
|33          |2025-01-01 00:00:00|18    |
|36          |2025-01-01 00:00:00|6     |
|37          |2025-01-01 00:00:00|10    |
|39          |2025-01-01 00:00:00|1     |
|40          |2025-01-01 00:00:00|

In [60]:
print(f"Zone-Stunden-Kombinationen: {hourly_demand.count():,}")

[Stage 212:===================================>                     (5 + 3) / 8]

Zone-Stunden-Kombinationen: 96,518


In [61]:
hourly_demand.select(
    F.min("demand").alias("min_demand"),
    F.max("demand").alias("max_demand"),
    F.avg("demand").alias("avg_demand")
).show()

[Stage 228:===================================>                     (5 + 3) / 8]

+----------+----------+------------------+
|min_demand|max_demand|        avg_demand|
+----------+----------+------------------+
|         1|       936|34.922087071841524|
+----------+----------+------------------+



In [62]:
zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/raw/taxi_zone_lookup.csv")
)

zones.show(10, truncate=False)
zones.printSchema()

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 10 rows
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable 

In [63]:
print(f"Anzahl Zonen: {zones.count()}")

Anzahl Zonen: 265


In [64]:
zones.select(
    F.min("LocationID").alias("min_id"),
    F.max("LocationID").alias("max_id"),
    F.countDistinct("LocationID").alias("distinct_ids")
).show()

+------+------+------------+
|min_id|max_id|distinct_ids|
+------+------+------------+
|     1|   265|         265|
+------+------+------------+



In [65]:
hours = spark.sql("""
    SELECT explode(
        sequence(
            timestamp('2025-01-01 00:00:00'),
            timestamp('2025-01-31 23:00:00'),
            interval 1 hour
        )
    ) AS pickup_hour
""")

print(f"Anzahl Stunden: {hours.count()}")

Anzahl Stunden: 744


In [66]:
zone_hours = (
    zones
    .select(
        "LocationID",
        "Borough",
        "Zone",
        "service_zone"
    )
    .crossJoin(hours)
)

print(f"Zone-Hour-Kombinationen: {zone_hours.count():,}")

Zone-Hour-Kombinationen: 197,160


In [67]:
demand_complete = (
    zone_hours
    .join(
        hourly_demand,
        (zone_hours["LocationID"] == hourly_demand["PULocationID"]) &
        (zone_hours["pickup_hour"] == hourly_demand["pickup_hour"]),
        "left"
    )
    .select(
        zone_hours["LocationID"],
        zone_hours["Borough"],
        zone_hours["Zone"],
        zone_hours["service_zone"],
        zone_hours["pickup_hour"],
        hourly_demand["demand"]
    )
)

In [68]:
demand_complete = demand_complete.fillna(
    {"demand": 0}
)

In [69]:
print(f"Zeilen: {demand_complete.count():,}")

Zeilen: 197,160


In [70]:
demand_complete.select(
    F.min("demand").alias("min_demand"),
    F.max("demand").alias("max_demand"),
    F.avg("demand").alias("avg_demand"),
    F.sum(
        (F.col("demand") == 0).cast("int")
    ).alias("zero_demand_hours")
).show()

+----------+----------+------------------+-----------------+
|min_demand|max_demand|        avg_demand|zero_demand_hours|
+----------+----------+------------------+-----------------+
|         0|       936|17.095810509231082|           100642|
+----------+----------+------------------+-----------------+



In [71]:
total_demand = demand_complete.agg(
    F.sum("demand").alias("total_demand")
).first()["total_demand"]

print(f"Summe Demand:          {total_demand:,}")
print(f"Valide Pickup-Records: {demand_valid.count():,}")

Summe Demand:          3,370,610


Valide Pickup-Records: 3,370,610


## 4. Feature Engineering

Temporal and spatial features are derived from the hourly demand dataset
for use in the demand-prediction model.

In [72]:
features = (
    demand_complete
    .withColumn("hour", F.hour("pickup_hour"))
    .withColumn("day_of_week", F.dayofweek("pickup_hour"))
    .withColumn("day_of_month", F.dayofmonth("pickup_hour"))
    .withColumn(
        "is_weekend",
        F.when(
            F.dayofweek("pickup_hour").isin([1, 7]),
            1
        ).otherwise(0)
    )
)

In [73]:
features.select(
    "LocationID",
    "Zone",
    "pickup_hour",
    "hour",
    "day_of_week",
    "day_of_month",
    "is_weekend",
    "demand"
).orderBy(
    "pickup_hour",
    "LocationID"
).show(20, truncate=False)

+----------+-----------------------+-------------------+----+-----------+------------+----------+------+
|LocationID|Zone                   |pickup_hour        |hour|day_of_week|day_of_month|is_weekend|demand|
+----------+-----------------------+-------------------+----+-----------+------------+----------+------+
|1         |Newark Airport         |2025-01-01 00:00:00|0   |4          |1           |0         |0     |
|2         |Jamaica Bay            |2025-01-01 00:00:00|0   |4          |1           |0         |0     |
|3         |Allerton/Pelham Gardens|2025-01-01 00:00:00|0   |4          |1           |0         |0     |
|4         |Alphabet City          |2025-01-01 00:00:00|0   |4          |1           |0         |28    |
|5         |Arden Heights          |2025-01-01 00:00:00|0   |4          |1           |0         |0     |
|6         |Arrochar/Fort Wadsworth|2025-01-01 00:00:00|0   |4          |1           |0         |0     |
|7         |Astoria                |2025-01-01 00:00:00

In [74]:
import math

features = (
    features
    .withColumn(
        "hour_sin",
        F.sin(2 * math.pi * F.col("hour") / 24)
    )
    .withColumn(
        "hour_cos",
        F.cos(2 * math.pi * F.col("hour") / 24)
    )
)

In [75]:
features = (
    features
    .withColumn(
        "dow_sin",
        F.sin(2 * math.pi * (F.col("day_of_week") - 1) / 7)
    )
    .withColumn(
        "dow_cos",
        F.cos(2 * math.pi * (F.col("day_of_week") - 1) / 7)
    )
)

In [76]:
features.select(
    "pickup_hour",
    "hour",
    "hour_sin",
    "hour_cos",
    "day_of_week",
    "dow_sin",
    "dow_cos"
).distinct().orderBy("pickup_hour").show(30, truncate=False)

+-------------------+----+----------------------+-----------------------+-----------+-------------------+-------------------+
|pickup_hour        |hour|hour_sin              |hour_cos               |day_of_week|dow_sin            |dow_cos            |
+-------------------+----+----------------------+-----------------------+-----------+-------------------+-------------------+
|2025-01-01 00:00:00|0   |0.0                   |1.0                    |4          |0.43388373911755823|-0.900968867902419 |
|2025-01-01 01:00:00|1   |0.25881904510252074   |0.9659258262890683     |4          |0.43388373911755823|-0.900968867902419 |
|2025-01-01 02:00:00|2   |0.49999999999999994   |0.8660254037844387     |4          |0.43388373911755823|-0.900968867902419 |
|2025-01-01 03:00:00|3   |0.7071067811865475    |0.7071067811865476     |4          |0.43388373911755823|-0.900968867902419 |
|2025-01-01 04:00:00|4   |0.8660254037844386    |0.5000000000000001     |4          |0.43388373911755823|-0.9009688679

In [77]:
from pyspark.sql.window import Window

zone_window = (
    Window
    .partitionBy("LocationID")
    .orderBy("pickup_hour")
)

In [78]:
features = (
    features
    .withColumn(
        "lag_1h",
        F.lag("demand", 1).over(zone_window)
    )
    .withColumn(
        "lag_24h",
        F.lag("demand", 24).over(zone_window)
    )
    .withColumn(
        "lag_168h",
        F.lag("demand", 168).over(zone_window)
    )
)

In [79]:
features.filter(
    F.col("LocationID") == 161
).select(
    "Zone",
    "pickup_hour",
    "demand",
    "lag_1h",
    "lag_24h",
    "lag_168h"
).orderBy(
    "pickup_hour"
).show(30, truncate=False)

+--------------+-------------------+------+------+-------+--------+
|Zone          |pickup_hour        |demand|lag_1h|lag_24h|lag_168h|
+--------------+-------------------+------+------+-------+--------+
|Midtown Center|2025-01-01 00:00:00|258   |NULL  |NULL   |NULL    |
|Midtown Center|2025-01-01 01:00:00|238   |258   |NULL   |NULL    |
|Midtown Center|2025-01-01 02:00:00|140   |238   |NULL   |NULL    |
|Midtown Center|2025-01-01 03:00:00|61    |140   |NULL   |NULL    |
|Midtown Center|2025-01-01 04:00:00|33    |61    |NULL   |NULL    |
|Midtown Center|2025-01-01 05:00:00|14    |33    |NULL   |NULL    |
|Midtown Center|2025-01-01 06:00:00|19    |14    |NULL   |NULL    |
|Midtown Center|2025-01-01 07:00:00|20    |19    |NULL   |NULL    |
|Midtown Center|2025-01-01 08:00:00|20    |20    |NULL   |NULL    |
|Midtown Center|2025-01-01 09:00:00|32    |20    |NULL   |NULL    |
|Midtown Center|2025-01-01 10:00:00|99    |32    |NULL   |NULL    |
|Midtown Center|2025-01-01 11:00:00|130   |99   

In [80]:
rolling_24h_window = (
    Window
    .partitionBy("LocationID")
    .orderBy("pickup_hour")
    .rowsBetween(-24, -1)
)

features = features.withColumn(
    "rolling_mean_24h",
    F.avg("demand").over(rolling_24h_window)
)

In [81]:
rolling_168h_window = (
    Window
    .partitionBy("LocationID")
    .orderBy("pickup_hour")
    .rowsBetween(-168, -1)
)

features = features.withColumn(
    "rolling_mean_168h",
    F.avg("demand").over(rolling_168h_window)
)

In [82]:
features = (
    features
    .withColumn(
        "rolling_mean_24h",
        F.avg("demand").over(rolling_24h_window)
    )
    .withColumn(
        "history_count_24h",
        F.count("demand").over(rolling_24h_window)
    )
    .withColumn(
        "rolling_mean_168h",
        F.avg("demand").over(rolling_168h_window)
    )
    .withColumn(
        "history_count_168h",
        F.count("demand").over(rolling_168h_window)
    )
)

In [83]:
features.filter(
    F.col("LocationID") == 161
).select(
    "pickup_hour",
    "demand",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "rolling_mean_24h",
    "history_count_24h",
    "rolling_mean_168h",
    "history_count_168h"
).orderBy("pickup_hour").show(30, truncate=False)

+-------------------+------+------+-------+--------+------------------+-----------------+------------------+------------------+
|pickup_hour        |demand|lag_1h|lag_24h|lag_168h|rolling_mean_24h  |history_count_24h|rolling_mean_168h |history_count_168h|
+-------------------+------+------+-------+--------+------------------+-----------------+------------------+------------------+
|2025-01-01 00:00:00|258   |NULL  |NULL   |NULL    |NULL              |0                |NULL              |0                 |
|2025-01-01 01:00:00|238   |258   |NULL   |NULL    |258.0             |1                |258.0             |1                 |
|2025-01-01 02:00:00|140   |238   |NULL   |NULL    |248.0             |2                |248.0             |2                 |
|2025-01-01 03:00:00|61    |140   |NULL   |NULL    |212.0             |3                |212.0             |3                 |
|2025-01-01 04:00:00|33    |61    |NULL   |NULL    |174.25            |4                |174.25         

In [84]:
model_data = features.filter(
    F.col("history_count_168h") == 168
)

In [85]:
train_data = model_data.filter(
    (F.col("pickup_hour") >= F.lit("2025-01-08 00:00:00")) &
    (F.col("pickup_hour") <  F.lit("2025-01-25 00:00:00"))
)

test_data = model_data.filter(
    (F.col("pickup_hour") >= F.lit("2025-01-25 00:00:00")) &
    (F.col("pickup_hour") <  F.lit("2025-02-01 00:00:00"))
)

In [86]:
print(f"Training rows: {train_data.count():,}")
print(f"Test rows:     {test_data.count():,}")

Training rows: 108,120
Test rows:     44,520


In [87]:
train_data.select(
    F.min("pickup_hour").alias("train_start"),
    F.max("pickup_hour").alias("train_end")
).show()

test_data.select(
    F.min("pickup_hour").alias("test_start"),
    F.max("pickup_hour").alias("test_end")
).show()

+-------------------+-------------------+
|        train_start|          train_end|
+-------------------+-------------------+
|2025-01-08 00:00:00|2025-01-24 23:00:00|
+-------------------+-------------------+

+-------------------+-------------------+
|         test_start|           test_end|
+-------------------+-------------------+
|2025-01-25 00:00:00|2025-01-31 23:00:00|
+-------------------+-------------------+



In [90]:
baseline = test_data.withColumn(
    "prediction",
    F.col("lag_24h").cast("double")
)

In [91]:
from pyspark.ml.evaluation import RegressionEvaluator

mae_evaluator = RegressionEvaluator(
    labelCol="demand",
    predictionCol="prediction",
    metricName="mae"
)

rmse_evaluator = RegressionEvaluator(
    labelCol="demand",
    predictionCol="prediction",
    metricName="rmse"
)

baseline_mae = mae_evaluator.evaluate(baseline)
baseline_rmse = rmse_evaluator.evaluate(baseline)

print(f"Baseline MAE:  {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")

Baseline MAE:  6.32
Baseline RMSE: 25.52


In [92]:
feature_columns = [
    "hour",
    "day_of_week",
    "day_of_month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "rolling_mean_24h",
    "rolling_mean_168h"
]

In [93]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

train_ml = assembler.transform(train_data)
test_ml = assembler.transform(test_data)

In [94]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="demand",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_ml)

In [95]:
predictions = rf_model.transform(test_ml)

In [96]:
rf_mae = mae_evaluator.evaluate(predictions)
rf_rmse = rmse_evaluator.evaluate(predictions)

print(f"Random Forest MAE:  {rf_mae:.2f}")
print(f"Random Forest RMSE: {rf_rmse:.2f}")

[Stage 503:==========================================>              (3 + 1) / 4]

Random Forest MAE:  4.61
Random Forest RMSE: 17.24


In [97]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.featureImportances.toArray()
}).sort_values("importance", ascending=False)

print(importance_df)

              feature  importance
8              lag_1h    0.351242
9             lag_24h    0.250108
10           lag_168h    0.248577
11   rolling_mean_24h    0.058846
12  rolling_mean_168h    0.039486
0                hour    0.012668
4            hour_sin    0.009588
5            hour_cos    0.008633
2        day_of_month    0.006129
7             dow_cos    0.005064
1         day_of_week    0.004299
6             dow_sin    0.003529
3          is_weekend    0.001831


In [98]:
for _, row in importance_df.iterrows():
    print(f"{row['feature']:<22} {row['importance']:.4f}")

lag_1h                 0.3512
lag_24h                0.2501
lag_168h               0.2486
rolling_mean_24h       0.0588
rolling_mean_168h      0.0395
hour                   0.0127
hour_sin               0.0096
hour_cos               0.0086
day_of_month           0.0061
dow_cos                0.0051
day_of_week            0.0043
dow_sin                0.0035
is_weekend             0.0018


In [99]:
spark.stop()